# Multivariate PCA Explorer

## Wisconsin Diagnostic Breast Cancer Dataset

This notebook analyzes high-dimensional structure using correlation analysis, principal component analysis (PCA), feature loadings, and an optional clustering comparison.

**Project question:**  
Can 30 tumor measurement features be reduced to two principal components while preserving meaningful diagnostic structure between malignant and benign cases?


## 1. Setup

The analysis uses the Wisconsin Diagnostic Breast Cancer dataset bundled with scikit-learn. The target label is kept separate from the numeric feature matrix so it is not included in correlation or PCA fitting.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", 50)


## 2. Load and Validate the Dataset

Inspect the number of observations, numeric features, class labels, data types, and missing values before analysis.


In [ ]:
cancer = load_breast_cancer(as_frame=True)

features = cancer.data.copy()

diagnosis = cancer.target.map({
    0: "Malignant",
    1: "Benign"
})

df = features.copy()
df["diagnosis"] = diagnosis

print("Observations:", df.shape[0])
print("Numeric features:", features.shape[1])
print("Classes:", sorted(df["diagnosis"].unique()))
print("Missing values:", int(df.isnull().sum().sum()))

df.head()


In [ ]:
feature_audit = pd.DataFrame({
    "dtype": features.dtypes.astype(str),
    "missing": features.isnull().sum(),
    "mean": features.mean(),
    "std": features.std()
})

feature_audit.head(10)


In [ ]:
class_balance = (
    df["diagnosis"]
    .value_counts()
    .rename_axis("diagnosis")
    .reset_index(name="count")
)

class_balance["share"] = class_balance["count"] / class_balance["count"].sum()
class_balance


## 3. Correlation Structure

A correlation heatmap helps identify groups of features that move together. Because the dataset has 30 numeric features, the full heatmap is displayed at a larger size for readability.


In [ ]:
corr = features.corr()

plt.figure(figsize=(16, 13))
sns.heatmap(
    corr,
    cmap="vlag",
    center=0,
    square=True,
    cbar_kws={"shrink": 0.7}
)
plt.title("Correlation Among Diagnostic Features")
plt.tight_layout()
plt.show()


### Strongest Off-Diagonal Correlation

Calculate the strongest absolute correlation outside the diagonal rather than selecting a pair by eye.


In [ ]:
corr_abs = corr.abs()

upper_triangle = corr_abs.where(
    np.triu(np.ones(corr_abs.shape), k=1).astype(bool)
)

strongest_pair = upper_triangle.stack().idxmax()
strongest_r = corr.loc[strongest_pair[0], strongest_pair[1]]

print("Strongest pair:", strongest_pair)
print(f"Correlation: {strongest_r:.3f}")


### Verify the Pair with a Scatterplot

The heatmap identifies the relationship; the scatterplot checks whether the relationship is actually linear and whether outliers or class structure are influencing the correlation.


In [ ]:
x_var, y_var = strongest_pair

plt.figure(figsize=(8, 5.5))
sns.scatterplot(
    data=df,
    x=x_var,
    y=y_var,
    hue="diagnosis",
    alpha=0.7
)

plt.title(f"{x_var} vs {y_var} (r = {strongest_r:.2f})")
plt.tight_layout()
plt.show()


### Correlation Interpretation

The correlation heatmap reveals substantial redundancy among several diagnostic measurements, especially among features describing tumor size and geometry. The strongest off-diagonal relationship is between **mean radius** and **mean perimeter**, with **r = 0.998**, indicating an almost perfect positive linear association. The verification scatterplot confirms that this relationship is consistently linear rather than being driven by a small number of outliers. The diagnostic classes occupy overlapping but visibly different regions: malignant cases tend to extend toward larger radius and perimeter values, while benign cases are concentrated more heavily at lower values.


## 4. Standardize the Features

PCA is sensitive to feature scale. The measurements use very different numeric ranges, so standardization is applied before PCA.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

scaled_summary = pd.DataFrame({
    "scaled_mean": X_scaled.mean(axis=0),
    "scaled_std": X_scaled.std(axis=0)
}, index=features.columns)

scaled_summary.head()


## 5. Principal Component Analysis

Fit PCA on the standardized 30-feature matrix and project each observation onto the first two principal components.


In [ ]:
pca = PCA(n_components=2)
pcs = pca.fit_transform(X_scaled)

explained = pd.DataFrame({
    "component": ["PC1", "PC2"],
    "explained_variance_ratio": pca.explained_variance_ratio_
})

explained["explained_variance_percent"] = (
    explained["explained_variance_ratio"] * 100
)

explained


In [ ]:
print(f"PC1 explains {pca.explained_variance_ratio_[0]:.1%} of the variance.")
print(f"PC2 explains {pca.explained_variance_ratio_[1]:.1%} of the variance.")
print(f"Together they explain {pca.explained_variance_ratio_.sum():.1%}.")


## 6. PCA Projection by Diagnostic Class

The known diagnosis is used only for coloring the PCA projection. It is not used to fit PCA.


In [ ]:
pca_df = pd.DataFrame(
    pcs,
    columns=["PC1", "PC2"],
    index=features.index
)

pca_df["diagnosis"] = diagnosis.values

plt.figure(figsize=(9, 6.5))
sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="diagnosis",
    alpha=0.75
)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} explained variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} explained variance)")
plt.title("PCA Projection of Diagnostic Measurements")
plt.tight_layout()
plt.show()


### PCA Structure Interpretation

After standardizing all 30 numeric features, **PC1 explains 44.3%** of the total variance and **PC2 explains 19.0%**, so the first two principal components preserve approximately **63.2%** of the standardized variance in a two-dimensional representation. The PCA projection shows substantial diagnostic structure: benign observations are concentrated mainly on the negative side of PC1, while malignant observations extend more strongly toward positive PC1 values. The classes still overlap near the center of the projection, so the separation is meaningful but not complete. Because PC1 and PC2 retain 63.2% rather than 100% of the variance, some structure present in the remaining principal components is necessarily omitted from this two-dimensional view.


## 7. Principal Component Loadings

Loadings show how strongly each original feature contributes to each principal component. Large absolute loadings are the most useful features for interpreting what PC1 and PC2 represent.


In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=features.columns,
    columns=["PC1", "PC2"]
)

loadings.head()


In [ ]:
top_pc1 = (
    loadings["PC1"]
    .abs()
    .sort_values(ascending=False)
    .head(10)
    .index
)

top_pc2 = (
    loadings["PC2"]
    .abs()
    .sort_values(ascending=False)
    .head(10)
    .index
)

print("Top PC1 contributors")
display(loadings.loc[top_pc1, ["PC1"]].sort_values("PC1"))

print("\nTop PC2 contributors")
display(loadings.loc[top_pc2, ["PC2"]].sort_values("PC2"))


In [ ]:
pc1_sorted = loadings["PC1"].sort_values()

plt.figure(figsize=(9, 8))
pc1_sorted.plot.barh()
plt.axvline(0, linewidth=0.8)
plt.xlabel("Loading")
plt.title("PC1 Feature Loadings")
plt.tight_layout()
plt.show()


In [ ]:
pc2_sorted = loadings["PC2"].sort_values()

plt.figure(figsize=(9, 8))
pc2_sorted.plot.barh()
plt.axvline(0, linewidth=0.8)
plt.xlabel("Loading")
plt.title("PC2 Feature Loadings")
plt.tight_layout()
plt.show()


### PC1 and PC2 Interpretation

**PC1:**  
The largest PC1 loadings are associated with **mean concave points, mean concavity, worst concave points, mean compactness, worst perimeter, worst concavity, worst radius, mean perimeter, worst area, and mean area**. These variables collectively describe tumor size together with shape irregularity and concavity. PC1 can therefore be interpreted as a broad **size-and-morphological-irregularity dimension**. Observations with larger PC1 scores tend to have larger and more irregular mass characteristics across several correlated measurements.

**PC2:**  
PC2 has strong positive contributions from **mean fractal dimension, fractal dimension error, worst fractal dimension, compactness error, and smoothness error**, while **mean radius, mean area, worst radius, worst area, mean perimeter, and worst perimeter** contribute in the opposite direction. PC2 therefore captures a contrast between **boundary complexity / fractal characteristics and overall tumor size**. This represents a different source of variation from PC1 rather than a second version of the same size-related pattern.

The sign of a PCA component is arbitrary; the interpretation depends on the relative loading pattern and relationships among variables, not on whether a loading is positive or negative in a particular run.


# Optional Extension — Classes vs. Clusters

PCA reveals low-dimensional structure but does not itself create clusters.

This extension applies K-Means to the standardized feature space and then compares the discovered clusters with the known diagnostic classes.

Because cluster IDs are arbitrary, a raw cluster label such as `0` does not automatically mean "Malignant" or "Benign".


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score, confusion_matrix
from scipy.optimize import linear_sum_assignment


In [ ]:
kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=20
)

cluster_labels = kmeans.fit_predict(X_scaled)
pca_df["cluster"] = cluster_labels.astype(str)

ari = adjusted_rand_score(cancer.target, cluster_labels)
silhouette = silhouette_score(X_scaled, cluster_labels)

print(f"Adjusted Rand Index: {ari:.3f}")
print(f"Silhouette Score: {silhouette:.3f}")


## Cluster Visualization in PCA Space

This plot colors the same PCA coordinates by discovered K-Means cluster instead of the known diagnosis.


In [ ]:
plt.figure(figsize=(9, 6.5))
sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="cluster",
    alpha=0.75
)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} explained variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} explained variance)")
plt.title("K-Means Clusters in PCA Space")
plt.tight_layout()
plt.show()


## Align Cluster IDs Before the Confusion Matrix

Cluster IDs are arbitrary. The Hungarian assignment algorithm finds the cluster-to-class mapping that maximizes agreement with the known labels before the aligned confusion matrix is calculated.


In [ ]:
raw_cm = confusion_matrix(cancer.target, cluster_labels)

row_ind, col_ind = linear_sum_assignment(-raw_cm)

cluster_to_class = {
    cluster_id: class_id
    for class_id, cluster_id in zip(row_ind, col_ind)
}

aligned_clusters = np.array(
    [cluster_to_class[c] for c in cluster_labels]
)

aligned_cm = confusion_matrix(cancer.target, aligned_clusters)

print("Cluster-to-class mapping:", cluster_to_class)
aligned_cm


In [ ]:
class_names = list(cancer.target_names)

cm_df = pd.DataFrame(
    aligned_cm,
    index=[f"Actual {name}" for name in class_names],
    columns=[f"Cluster mapped to {name}" for name in class_names]
)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_df,
    annot=True,
    fmt="d",
    cbar=False
)

plt.title("Aligned Class-Cluster Confusion Matrix")
plt.ylabel("True Diagnosis")
plt.xlabel("Aligned Cluster")
plt.tight_layout()
plt.show()

cm_df


### Class–Cluster Interpretation

K-Means clustering on the standardized 30-feature space shows meaningful agreement with the known diagnostic classes. The **Adjusted Rand Index is 0.671**, indicating substantial correspondence between the discovered clusters and the malignant/benign labels, while the **silhouette score is 0.345**, suggesting moderate rather than sharply separated cluster structure. After aligning the arbitrary cluster IDs to the diagnostic classes, the confusion matrix is **[[175, 37], [14, 343]]**. This means 175 of 212 malignant cases and 343 of 357 benign cases fall into the corresponding aligned clusters, for an overall aligned class–cluster agreement of approximately **91.0%**. Benign cases show stronger agreement (**96.1%**) than malignant cases (**82.5%**). These results are consistent with the PCA projection: the two diagnoses form meaningful structure, but an overlapping region remains.


# Final Summary

The analysis shows that the 30 diagnostic measurements contain substantial correlation and redundancy. The strongest observed relationship is between **mean radius and mean perimeter (r = 0.998)**. After standardization, **PC1 explains 44.3%** of the variance and **PC2 explains 19.0%**, so together they retain approximately **63.2%** of the standardized variance in two dimensions. The PCA projection reveals substantial but incomplete separation between malignant and benign cases. Based on the loading structure, PC1 primarily represents a combination of **tumor size and morphological irregularity**, while PC2 contrasts **fractal / boundary-complexity characteristics with size-related measurements**. As an extension, K-Means clustering produces an **ARI of 0.671**, a **silhouette score of 0.345**, and approximately **91.0% aligned class–cluster agreement**, confirming meaningful diagnostic structure while also showing that the groups are not perfectly separable.
